# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)]

**Lane: Refresh / Content Opportunity Scoring (Lane 2).** Same contract as `w03_data_contract.ipynb` and `w04_baseline_score.ipynb`: one row = one content page, features from **March 2026**, decision moment **2026-03-31**, label from **April 2026** (`is_declining` = April impressions < 80% of March, volume floor March ≥ 30 impressions).

This is the full-depth sibling of the data-contract notebook. It (1) builds the actual feature vector — numeric, flags, categorical — with every fill made explicit, (2) notes what each feature means and when it is knowable, (3) runs the **leakage hunt** on real warehouse rows: label-derived columns, future windows, a validation-design trick, and the missingness trap, and (4) lists what was excluded and why. The honest number survives; the leaks are shown and removed.

> Worked from `skills/hunting-leakage-and-validating/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Build the feature vector

One row = **one content page**. Everything below is computed from **February and March 2026** rows only — the April column exists solely to build the label, and the leakage hunt in section 3 proves no April value reaches the features. Fills are explicit, not hidden: `0` where a metric is genuinely zero, a `has_` flag where the metric is missing because the measurement did not exist.

In [1]:
# SETUP — the token stays in the runtime, never in a cell (this repo is public).
import os, sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore", message="IProgress not found")

if any(importlib.util.find_spec(m) is None for m in ("duckdb", "sklearn", "huggingface_hub")):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "duckdb", "scikit-learn", "huggingface_hub", "pandas", "numpy"], check=True)

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
pd.set_option("display.width", 200)

assert get_token(), ("No read token found. In Colab add a read-only HF_TOKEN as a Secret "
                     "(key panel) and rerun; the token is read from the runtime, not typed here.")

DS = "FlyRank/internship-warehouse"
P = {m: hf_hub_download(DS, f"fact_content_daily_performance/month={m}/data_0.parquet", repo_type="dataset")
     for m in ["2026-02", "2026-03", "2026-04"]}
DIM_CONTENT = hf_hub_download(DS, "dim_content.parquet", repo_type="dataset")
con = duckdb.connect()

DECISION_MOMENT = pd.Timestamp("2026-03-31")   # everything in the vector must be knowable here
FACT3 = "[" + ",".join(f"'{P[m]}'" for m in ["2026-02", "2026-03", "2026-04"]) + "]"

# One row per content page. Feb+Mar columns are the ONLY inputs the vector may touch;
# the April column exists solely to build the label further down.
raw = con.execute(f"""
    WITH f AS (
      SELECT content_hash_id,
             MAX(client_hash_id) AS client_hash_id,
             SUM(gsc_impressions) FILTER (WHERE month='2026-02' AND gsc_data_available IS TRUE) AS feb_impressions,
             SUM(gsc_impressions) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_impressions,
             SUM(gsc_clicks)      FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_clicks,
             -- Impression-weighted average position (both sums are integers -> reproducible).
             SUM(gsc_sum_position) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0) AS mar_sum_position,
             SUM(gsc_impressions)  FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0) AS mar_impr_with_position,
             SUM(ga4_engaged_sessions) FILTER (WHERE month='2026-03' AND ga4_data_available IS TRUE) AS mar_engaged,
             SUM(ga4_sessions)         FILTER (WHERE month='2026-03' AND ga4_data_available IS TRUE) AS mar_sessions,
             SUM(gsc_impressions) FILTER (WHERE month='2026-04' AND gsc_data_available IS TRUE) AS apr_impressions
      FROM read_parquet({FACT3})
      GROUP BY content_hash_id
    )
    SELECT f.*, d.content_created_date, d.content_type, d.search_volume, d.is_published, d.is_deleted
    FROM f LEFT JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
""").df()

for c in ["feb_impressions", "mar_impressions", "mar_clicks", "apr_impressions",
          "mar_engaged", "mar_sessions"]:
    raw[c] = raw[c].fillna(0)

raw["mar_avg_position"] = np.where(raw.mar_impr_with_position.fillna(0) > 0,
                                   raw.mar_sum_position / raw.mar_impr_with_position, np.nan)
raw["ctr_mar"]  = np.where(raw.mar_impressions > 0, raw.mar_clicks / raw.mar_impressions * 100, 0.0)
raw["momentum_feb_to_mar_pct"] = np.where(
    raw.feb_impressions > 0,
    (raw.mar_impressions - raw.feb_impressions) / raw.feb_impressions.replace(0, np.nan) * 100, np.nan)
raw["is_declining"] = (raw.apr_impressions < 0.8 * raw.mar_impressions).astype(int)

# Study population: visible in March (volume floor 30) and actionable (published, not deleted).
pop = raw[(raw.mar_impressions >= 30) & (raw.is_published == True) & (raw.is_deleted == False)].copy()

# ---- THE FEATURE VECTOR -----------------------------------------------------
fv = pop.copy()
fv["log_mar_impressions"] = np.log1p(fv.mar_impressions)
fv["has_clicks"]   = (fv.mar_clicks > 0).astype(int)
fv["has_feb_data"] = fv.feb_impressions.gt(0).astype(int)
fv["has_ga4"]      = (fv.mar_sessions > 0).astype(int)
fv["engagement_rate_mar"] = np.where(fv.mar_sessions > 0, fv.mar_engaged / fv.mar_sessions * 100, np.nan)
fv["content_type_cat"]    = fv.content_type.fillna("unknown").astype(str)

NUMERIC = ["log_mar_impressions", "ctr_mar", "mar_avg_position",
           "momentum_feb_to_mar_pct", "engagement_rate_mar"]
FLAGS   = ["has_clicks", "has_feb_data", "has_ga4"]
CATEGORICAL = ["content_type_cat"]
FEATURES8 = NUMERIC + FLAGS

print(f"pages with any March search data : {len(raw):,}")
print(f"study population                 : {len(pop):,}")
print(f"base rate P(is_declining)        : {pop.is_declining.mean():.4f}")
print("\nfeature vector head (8 inputs + one categorical + the label):")
print(fv[["content_hash_id", "client_hash_id"] + FEATURES8 + ["content_type_cat", "is_declining"]]
      .head().to_string(index=False))

pages with any March search data : 380,147
study population                 : 125,573
base rate P(is_declining)        : 0.5181

feature vector head (8 inputs + one categorical + the label):
         content_hash_id          client_hash_id  log_mar_impressions  ctr_mar  mar_avg_position  momentum_feb_to_mar_pct  engagement_rate_mar  has_clicks  has_feb_data  has_ga4 content_type_cat  is_declining
content_019308230fe21954 client_e547b89c05043229             8.219865 0.053865         21.468893               104.460352                  NaN           1             1        0  keyword article             0
content_953377c51dada952 client_e547b89c05043229             6.632002 0.000000         27.233509               166.901408                  NaN           0             1        0  keyword article             0
content_af11cb477a8cd182 client_e547b89c05043229             6.371612 0.171233          9.794521               160.714286             0.000000           1             1        1  key

## 2. Feature notes — meaning, missing, categorical, available-when?

| Feature | Meaning | Fill / missing | Available when? |
|---|---|---|---|
| `log_mar_impressions` | log1p of March GSC impressions | 0 → none missing | at `2026-03-31`: March's daily rows are written |
| `ctr_mar` | March clicks / impressions ×100 (a ×100 percentage) | 0 when no clicks | at `2026-03-31`: both totals final |
| `mar_avg_position` | impression-weighted March position index | NaN → no day with a real position (>0) | at `2026-03-31`: month's positions stop accruing |
| `momentum_feb_to_mar_pct` | Feb→Mar impression change, % | NaN when no Feb data (page registered mid-window) — 16% of rows | at `2026-03-31`: both months closed |
| `engagement_rate_mar` | March GA4 engaged ÷ sessions, on `ga4_data_available IS TRUE` rows | NaN when no GA4 mark — 51% of rows | at `2026-03-31`: GA4 rows only; the flag is three-valued |
| `has_clicks` | 1 when March clicks > 0 | never missing | at `2026-03-31` |
| `has_feb_data` | 1 when the page has February rows | never missing | at `2026-03-31` |
| `has_ga4` | 1 when March GA4 sessions > 0 | never missing | at `2026-03-31` |
| `content_type_cat` | `keyword article` / `comparison article` / `feedly article` | filled `unknown` (0% missing) | static metadata, knowable always |

In [2]:
# Missing values, made explicit — each gap is a MEASUREMENT gap, not a zero.
miss = fv[NUMERIC + FLAGS + ["content_type_cat"]].isna().mean()
print("NaN share per column (missingness is a signal, not noise):")
print(miss.round(4).to_string())

print("\nwhy each gap exists:")
print("  momentum_feb_to_mar_pct  -> page had no February rows: it registered mid-window.")
print("  engagement_rate_mar      -> no GA4 mark in March: the client's GA4 tracking started later")
print("                              (availability flag is three-valued: TRUE / FALSE / NULL).")
print("  mar_avg_position         -> no March day with a real position (>0).")
print("  content_type_cat         -> never missing after the 'unknown' fill.")
print("\nRule used here: keep the gap as NaN, add a has_ flag. Blind fillna(0) would encode")
print("'measurement absent' as 'value zero' — attack C shows why that misleads.")

NaN share per column (missingness is a signal, not noise):
log_mar_impressions        0.0000
ctr_mar                    0.0000
mar_avg_position           0.0000
momentum_feb_to_mar_pct    0.1634
engagement_rate_mar        0.5063
has_clicks                 0.0000
has_feb_data               0.0000
has_ga4                    0.0000
content_type_cat           0.0000

why each gap exists:
  momentum_feb_to_mar_pct  -> page had no February rows: it registered mid-window.
  engagement_rate_mar      -> no GA4 mark in March: the client's GA4 tracking started later
                              (availability flag is three-valued: TRUE / FALSE / NULL).
  mar_avg_position         -> no March day with a real position (>0).
  content_type_cat         -> never missing after the 'unknown' fill.

Rule used here: keep the gap as NaN, add a has_ flag. Blind fillna(0) would encode
'measurement absent' as 'value zero' — attack C shows why that misleads.


## 3. The leakage hunt

Four attacks, each with a number on the same 125k-page population. The honest model: a logistic regression on the 8-feature vector, random 70/30 stratified split. Anything that needs April to compute — or any validation trick that lets the model meet its own clients again — is named, shown, and removed. The number that survives is the only one reported onward.

| Attack | What it adds | What it proves |
|---|---|---|
| **A. Future-window column** | a March→April change % as a "feature" | a column knowable only after 31 Mar jumps the score toward perfect |
| **B. Validation design** | random split vs client-grouped holdout | pages from one client are too similar; random split overstates skill |
| **C. Missingness trap** | filling "no GA4" with 0 | the fill smuggles a `has_ga4` flag into a fake engagement reading |
| **D. Label-copy / product-flag** | the label's own rule rebuilt as a "feature" | feeding the answer (or a flag that encodes it) is circular |

In [3]:
# ATTACK A — a column knowable only AFTER the decision moment.
X = fv[FEATURES8].fillna(0).values
y = fv.is_declining.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)
honest = roc_auc_score(yte, LogisticRegression(max_iter=1000)
                       .fit(sc.transform(Xtr), ytr).predict_proba(sc.transform(Xte))[:, 1])

# The trap: this column needs April to exist, but it sits in the table like any other.
leak = fv.copy()
leak["mar_to_apr_change_pct"] = np.where(leak.mar_impressions > 0,
                                         (leak.apr_impressions - leak.mar_impressions) / leak.mar_impressions * 100, 0.0)
Xl = leak[FEATURES8 + ["mar_to_apr_change_pct"]].fillna(0).values
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.3, random_state=0, stratify=y)
scl = StandardScaler().fit(Xltr)
leaked = roc_auc_score(ylte, LogisticRegression(max_iter=1000)
                       .fit(scl.transform(Xltr), yltr).predict_proba(scl.transform(Xlte))[:, 1])

print(f"honest 8-feature model                    ROC AUC = {honest:.4f}")
print(f"+ one March-to-April change column (future)  ROC AUC = {leaked:.4f}")
print(f"\nThat column is derived from the LABEL window (April). It is dropped — the honest")
print(f"number is {honest:.4f}, not the {leaked:.4f}.")

honest 8-feature model                    ROC AUC = 0.6191
+ one March-to-April change column (future)  ROC AUC = 1.0000

That column is derived from the LABEL window (April). It is dropped — the honest
number is 0.6191, not the 1.0000.


In [4]:
# ATTACK B — validation design: random split vs client-grouped holdout.
def grouped_auc(Xdf, ydf, clients, n_splits=4):
    aucs = []
    for tr, te in GroupKFold(n_splits=n_splits).split(Xdf, ydf, groups=clients):
        sc = StandardScaler().fit(Xdf.values[tr])
        m = LogisticRegression(max_iter=1000).fit(sc.transform(Xdf.values[tr]), ydf.values[tr])
        aucs.append(roc_auc_score(ydf.values[te], m.predict_proba(sc.transform(Xdf.values[te]))[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))

Xdf = fv[FEATURES8].fillna(0)
g_auc, g_std = grouped_auc(Xdf, fv.is_declining, fv.client_hash_id)
print(f"client-grouped holdout (4 folds)  ROC AUC = {g_auc:.4f} +/- {g_std:.4f}")
print(f"random 70/30 split (attack A)     ROC AUC = {honest:.4f}")
print(f"\nThe random split overstates skill by ~{honest - g_auc:.3f}: pages from the same client")
print("share patterns, so the model memorizes clients it then 'meets again' in the test set.")
print("Client-grouped validation is the honest design for this lane.")

client-grouped holdout (4 folds)  ROC AUC = 0.5971 +/- 0.0581
random 70/30 split (attack A)     ROC AUC = 0.6191

The random split overstates skill by ~0.022: pages from the same client
share patterns, so the model memorizes clients it then 'meets again' in the test set.
Client-grouped validation is the honest design for this lane.


In [5]:
# ATTACK C — the missingness trap: filling "no GA4" with 0.
print("decline rate by has_ga4 (the signal a blind fill hides):")
print(fv.groupby("has_ga4").agg(n=("is_declining", "size"),
                                decline_rate=("is_declining", "mean")).round(3).to_string())
print(f"\nPages WITHOUT a GA4 mark decline at a different rate than pages with GA4 — the two")
print("populations are not the same. Filling engagement_rate_mar = 0 for them would read as")
print("'zero engagement' when it actually means 'no GA4 measurement' (51% of rows).")
print("The vector keeps engagement NaN and adds the has_ga4 flag, so the signal is explicit,")
print("and no fill ever turns absence into a number.")

decline rate by has_ga4 (the signal a blind fill hides):
             n  decline_rate
has_ga4                     
0        63582         0.555
1        61991         0.480

Pages WITHOUT a GA4 mark decline at a different rate than pages with GA4 — the two
populations are not the same. Filling engagement_rate_mar = 0 for them would read as
'zero engagement' when it actually means 'no GA4 measurement' (51% of rows).
The vector keeps engagement NaN and adds the has_ga4 flag, so the signal is explicit,
and no fill ever turns absence into a number.


In [6]:
# ATTACK D — the label's own rule rebuilt as a "feature" (the circular / product-flag trap).
# The starter label is is_declining_label = (trend_direction == "down"), computed from impressions.
# Rebuilding that same rule on the label window and feeding it in copies the answer.
label_copy = (fv.apr_impressions < 0.8 * fv.mar_impressions).astype(int).values
print(f"label-copy 'feature' ROC AUC = {roc_auc_score(y, label_copy):.4f}   (it IS the label)")
print("\nFlyRank's own product flags (health_score, priority_score, refresh flags) are NOT in this")
print("dataset, so the only 'product flags' that could leak here are ones we rebuild ourselves —")
print("and rebuilding the label's rule is circular, exactly as shown above. No product-coded")
print("flag is an input to the vector.")

label-copy 'feature' ROC AUC = 1.0000   (it IS the label)

FlyRank's own product flags (health_score, priority_score, refresh flags) are NOT in this
dataset, so the only 'product flags' that could leak here are ones we rebuild ourselves —
and rebuilding the label's rule is circular, exactly as shown above. No product-coded
flag is an input to the vector.


In [7]:
# Verdict: what survives, what is removed.
print(f"{'model / column':44s} {'ROC AUC':>20s}  verdict")
print(f"{'honest 8-feature model (random split)':44s} {honest:>20.4f}  KEEP (optimistic design)")
print(f"{'+ future-window change column':44s} {leaked:>20.4f}  REMOVE (label window)")
print(f"{'client-grouped holdout (honest design)':44s} {g_auc:>15.4f} +/- {g_std:.4f}  KEEP (taken forward)")
print(f"{'label-copy feature':44s} {'1.0000':>20s}  REMOVE (circular)")
print(f"\nHonest number taken forward: client-grouped ROC AUC = {g_auc:.4f}, on the 8-feature")
print(f"vector with no April-derived column. The trap score ({leaked:.4f}) is never reported.")

model / column                                            ROC AUC  verdict
honest 8-feature model (random split)                      0.6191  KEEP (optimistic design)
+ future-window change column                              1.0000  REMOVE (label window)
client-grouped holdout (honest design)                0.5971 +/- 0.0581  KEEP (taken forward)
label-copy feature                                         1.0000  REMOVE (circular)

Honest number taken forward: client-grouped ROC AUC = 0.5971, on the 8-feature
vector with no April-derived column. The trap score (1.0000) is never reported.


## 4. What I excluded and why

| Excluded | Why (one line) |
|---|---|
| `fact_content_query_90d` | its fixed 90-day window overlaps March+April (the label months) → leaks |
| April / future-window columns | they contain the answer; attack A shows the jump |
| `trend_direction` / `trend_pct` buckets | the starter's label source; rebuilding them is circular (attack D) |
| `content_updated_date` | snapshot-as-of-build, runs to 2026-07-06 → "days since update" unknowable for most rows |
| `is_published` / `is_deleted` as features | same snapshot family; used only to keep pages an editor can act on |
| `client_hash_id`, `content_hash_id` as features | join keys / pseudonyms, grouping and splits only |
| product flags (`health_score`, `priority_score`, …) | not shipped in this dataset; if rebuilt, circular by construction |
| raw query / URL / title fields | private; not shipped in this dataset |

In [8]:
# The exclusion list, restated in code so it stays checkable.
EXCLUDED = {
    "fact_content_query_90d":        "fixed 90-day window overlaps the label months -> leaks",
    "apr_impressions (April)":       "future column; contains the answer",
    "trend_direction / trend_pct":   "starter label source; rebuilding is circular",
    "content_updated_date":          "snapshot-as-of-build; unknowable at the decision moment",
    "is_published / is_deleted":     "snapshot family; actionability filter only, never a feature",
    "client_hash_id/content_hash_id":"join keys; grouping and splits only",
    "product flags (health/priority)":"not in the data; circular if rebuilt",
    "raw query / URL / title":       "private; not in the dataset",
}
for k, v in EXCLUDED.items():
    print(f"  {k:42s} -> {v}")

# The delivered vector must carry no label-window column.
leak_cols = [c for c in fv.columns if "apr" in c.lower() or "declin" in c.lower() or "mar_to_apr" in c.lower()]
print(f"\nlabel-window columns in the feature vector: {leak_cols}")
print("(apr_impressions exists in the base frame to BUILD the label — it is not in FEATURES8, "
      "and it is dropped before any model or queue in later weeks.)")

  fact_content_query_90d                     -> fixed 90-day window overlaps the label months -> leaks
  apr_impressions (April)                    -> future column; contains the answer
  trend_direction / trend_pct                -> starter label source; rebuilding is circular
  content_updated_date                       -> snapshot-as-of-build; unknowable at the decision moment
  is_published / is_deleted                  -> snapshot family; actionability filter only, never a feature
  client_hash_id/content_hash_id             -> join keys; grouping and splits only
  product flags (health/priority)            -> not in the data; circular if rebuilt
  raw query / URL / title                    -> private; not in the dataset

label-window columns in the feature vector: ['apr_impressions', 'is_declining']
(apr_impressions exists in the base frame to BUILD the label — it is not in FEATURES8, and it is dropped before any model or queue in later weeks.)


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous hash IDs
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] **Section 1** builds the feature vector — numeric, flags, one categorical — with every fill explicit
- [x] **Section 2** gives each feature a meaning, its missing-handling, and an "available when?" line
- [x] **Section 3** hunts leakage on real warehouse rows: future-window column (A), validation design (B),
      missingness trap (C), label-copy/product-flag (D) — each with a number, leaks shown then removed
- [x] **Section 4** lists what was excluded and why, and proves no label-window column is in the vector
- [x] Honest number kept: client-grouped ROC AUC; the trap score is never reported onward
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.